In [1]:
'''
# Install unrar
!apt-get install unrar

# Unzip the rar files
!unrar x "/content/test 1.rar"
!unrar x "/content/train 1.rar"
'''

'\n# Install unrar\n!apt-get install unrar\n\n# Unzip the rar files\n!unrar x "/content/test 1.rar"\n!unrar x "/content/train 1.rar"\n'

In [2]:
# pip install tensorflow opencv-python

In [3]:
# Import the necessary libraries
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import time

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
from tensorflow.data import AUTOTUNE
from tensorflow.keras import layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

from scipy.linalg import sqrtm

import cv2

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load the training metadata
train_df = pd.read_csv('./Train1/processed_metadata_train.csv')

In [5]:
# Check 10 samples
train_df.sample(10)

,new_photo_id,label,original_photo_id,augmented
22189,outside_22190_n6pTHg6JLgnJYHuLKwhLfw,outside,n6pTHg6JLgnJYHuLKwhLfw,False
12726,inside_12727_H-6ToNPf9CSvLq9NXB3K4w,inside,H-6ToNPf9CSvLq9NXB3K4w,False
14016,inside_14017_hTFFfpmVH7KTxTV4DHm1qg,inside,hTFFfpmVH7KTxTV4DHm1qg,False
23093,outside_23094_JckKcYSA3vZKDAddIzc7Fg,outside,JckKcYSA3vZKDAddIzc7Fg,False
19254,menu_19255_aug,menu,QYTWP-F312Wr8NaNBAyROA,True
2186,drink_2187_4DLmlD-vNPAMRU7lIQkSEg,drink,4DLmlD-vNPAMRU7lIQkSEg,False
8492,food_8493_xJvGf1EbwA8P_KsRbfusFg,food,xJvGf1EbwA8P_KsRbfusFg,False
12875,inside_12876_EfL6zUvg4J-6SeF1RiXsWg,inside,EfL6zUvg4J-6SeF1RiXsWg,False
23492,outside_23493_OLW6hvN91MdmDSA_JomyDQ,outside,OLW6hvN91MdmDSA_JomyDQ,False
9834,food_9835_rFt7-e_iBZoRBppebbSGug,food,rFt7-e_iBZoRBppebbSGug,False


In [6]:
# Check info
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   new_photo_id       25000 non-null  object
 1   label              25000 non-null  object
 2   original_photo_id  25000 non-null  object
 3   augmented          25000 non-null  bool  
dtypes: bool(1), object(3)
memory usage: 610.5+ KB


In [7]:
# Load the test metadata
test_df = pd.read_csv('./test/processed_metadata.csv')

In [8]:
# Check 10 samples
test_df.sample(10)

,new_photo_id,label,original_photo_id,augmented
1274,food_1275_iWZlU-ntFNShq6-tc6X0GA,food,iWZlU-ntFNShq6-tc6X0GA,False
4702,outside_4703_UbvCWCUbxtOEkOvl2mI_Tg,outside,UbvCWCUbxtOEkOvl2mI_Tg,False
987,drink_988_JOiArqQVjpGkrsN369eI9w,drink,JOiArqQVjpGkrsN369eI9w,False
3754,menu_3755_aug,menu,Z3VG7SqloTEHIqSeRsPxzg,True
3753,menu_3754_aug,menu,NapcYkQqMvrPNpjRsV5cPg,True
4392,outside_4393_gvVCZQQyyKWinUieea0iUw,outside,gvVCZQQyyKWinUieea0iUw,False
1651,food_1652_hdYU-TtKaWne5Mfm6lt5cA,food,hdYU-TtKaWne5Mfm6lt5cA,False
2747,inside_2748_hN9R_FLSt1fTq6M3cUpHlQ,inside,hN9R_FLSt1fTq6M3cUpHlQ,False
3055,menu_3056_tZPwxpKqbQFuDIVsGhaPsQ,menu,tZPwxpKqbQFuDIVsGhaPsQ,False
213,drink_214_E0RCxWZPPqjO4rRLcf6ppw,drink,E0RCxWZPPqjO4rRLcf6ppw,False


In [9]:
# Check info
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   new_photo_id       5000 non-null   object
 1   label              5000 non-null   object
 2   original_photo_id  5000 non-null   object
 3   augmented          5000 non-null   bool  
dtypes: bool(1), object(3)
memory usage: 122.2+ KB


## Filter Valid Photos

In [10]:
# Remove all datasets with "Menu" label
train_df = train_df[train_df['label'] != 'menu'].reset_index(drop=True)

In [11]:
# Remove all datasets with "Menu" label
test_df = test_df[test_df['label'] != 'menu'].reset_index(drop=True)

In [12]:
# Image directory
img_dir_train = './Train1/processed_photos_train'
img_dir_test = './test/processed_photos'

In [13]:
# Count the items in image directories
num_items_train = len(os.listdir(img_dir_train))
num_items_test = len(os.listdir(img_dir_test))

print(f"Number of items in {img_dir_train}: {num_items_train}")
print(f"Number of items in {img_dir_test}: {num_items_test}")

Number of items in ./Train1/processed_photos_train: 25000
Number of items in ./test/processed_photos: 5000


In [14]:
def filter_valid_photos(df, image_dir):
    valid_indices = []
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Validating photos"):
        photo_id = row['new_photo_id']
        img_path = os.path.join(image_dir, f'{photo_id}.jpg')
        try:
            _ = tf.keras.preprocessing.image.load_img(img_path)
            valid_indices.append(i)
        except Exception as e:
            continue
    return df.loc[valid_indices].reset_index(drop=True)

# Clean metadata
start_time = time.time()
train_df = filter_valid_photos(train_df, image_dir=img_dir_train)
test_df = filter_valid_photos(test_df, image_dir=img_dir_test)
print(f'After cleaning, training metadata has {len(train_df)} rows. Took {time.time()-start_time:.2f}s.')
print(f'After cleaning, test metadata has {len(test_df)} rows. Took {time.time()-start_time:.2f}s.')

Validating photos: 100%|██████████████████████████████████████████████████████████| 4000/4000 [00:04<00:00, 913.55it/s]

After cleaning, training metadata has 20000 rows. Took 29.92s.
After cleaning, test metadata has 4000 rows. Took 29.93s.


In [15]:
train_df.head()

,new_photo_id,label,original_photo_id,augmented
0,drink_1_1u4I3V3fhRDDfLHRbDwO9w,drink,1u4I3V3fhRDDfLHRbDwO9w,False
1,drink_2_kCob6wOKqwm6hXQ-WLYCOA,drink,kCob6wOKqwm6hXQ-WLYCOA,False
2,drink_3_0eDzm3uqdjvj1srynbO0uw,drink,0eDzm3uqdjvj1srynbO0uw,False
3,drink_4_tOATeq906QAXmmB8jWouNw,drink,tOATeq906QAXmmB8jWouNw,False
4,drink_5_Vc0E0fGqglH0Bj8fpfR29g,drink,Vc0E0fGqglH0Bj8fpfR29g,False


In [16]:
test_df.head()

,new_photo_id,label,original_photo_id,augmented
0,drink_1_8VfJq1vTpMh6IMKhtbsqHg,drink,8VfJq1vTpMh6IMKhtbsqHg,False
1,drink_2_3LGq6soWgPCWeVqaJCqCnw,drink,3LGq6soWgPCWeVqaJCqCnw,False
2,drink_3_YoIAYWkm-2Di1c46gyJUXQ,drink,YoIAYWkm-2Di1c46gyJUXQ,False
3,drink_4_NwjHP2DiGj3xhCd8PnYcgg,drink,NwjHP2DiGj3xhCd8PnYcgg,False
4,drink_5_1XR1orXOs9DcwWwbep9USw,drink,1XR1orXOs9DcwWwbep9USw,False


# 4. cGAN

## Encode labels: [food, drink, inside, outside]

In [17]:
# Initialize labels
LABELS = ['food', 'drink', 'inside', 'outside']

In [18]:
le = LabelEncoder()
train_df['label_encoded'] = le.fit_transform(train_df['label'])
test_df['label_encoded'] = le.fit_transform(test_df['label'])

encoder = OneHotEncoder(sparse_output=False)
onehot_labels_train = encoder.fit_transform(train_df[['label_encoded']])
onehot_labels_test = encoder.transform(test_df[['label_encoded']])

print('Label mapping:', dict(zip(le.classes_, le.transform(le.classes_))))

# Test small sample
sample_train_df = train_df.sample(5)
sample_test_df = test_df.sample(5)
print(sample_train_df[['new_photo_id', 'label', 'label_encoded']])
print('\n')
print(sample_test_df[['new_photo_id', 'label', 'label_encoded']])

Label mapping: {'drink': 0, 'food': 1, 'inside': 2, 'outside': 3}
                               new_photo_id    label  label_encoded
2106      drink_2107_lYnbjZHh8VjxWYQUHJGbCA    drink              0
3737      drink_3738_-FcpVDNCx0aIsRRZCVwdQQ    drink              0
10515   inside_10516_TjwgXWNAwGI7FDFayUhLIA   inside              2
10189   inside_10190_rBqRTE7hkMAFgwk2q3jLVw   inside              2
16105  outside_21106_NbvY-4GaAa8KvX8PFnzQaQ  outside              3


                             new_photo_id    label  label_encoded
44        drink_45_rixO_iuwKwteHT2j6f31Gg    drink              0
3361  outside_4362_VNTTge-kx5mfwS3oFd3p4g  outside              3
1927     food_1928_-jxRC49iFen_BICFbAQ9yQ     food              1
3495  outside_4496_HKMnPzknZn70TsIAuSLjkQ  outside              3
1372     food_1373_qUbmd3nOtxDReWas14-CXA     food              1


## Load sample images to test pipeline

In [19]:
def load_images(photo_ids, directory=img_dir_train, target_size=(64,64)):
    images = []
    for pid in tqdm(photo_ids, desc="Loading images"):
        try:
            path = os.path.join(directory, f'{pid}.jpg')
            img = tf.keras.preprocessing.image.load_img(path, target_size=target_size)
            img = tf.keras.preprocessing.image.img_to_array(img) / 255.0
            images.append(img)
        except Exception as e:
            print(f'Could not load {path}: {e}')
            continue
    return np.array(images)

# Load a small sample for quick test
start_time = time.time()
train_sample = load_images(train_df['new_photo_id'].sample(5))
test_sample = load_images(test_df['new_photo_id'].sample(5), directory=img_dir_test)
print(f'Loaded {len(train_sample)} sample images in {time.time()-start_time:.2f}s')
print(f'Loaded {len(test_sample)} sample images in {time.time()-start_time:.2f}s')

Loading images: 100%|██████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 1252.40it/s]

Loaded 5 sample images in 0.03s
Loaded 5 sample images in 0.03s


## Load images from the main (Training and Test) dataframes

In [20]:
start_time = time.time()
train_array = load_images(train_df['new_photo_id'])
test_array = load_images(test_df['new_photo_id'], directory=img_dir_test)
print(f'Loaded {len(train_array)} total images in {time.time()-start_time:.2f}s')
print(f'Loaded {len(test_array)} total images in {time.time()-start_time:.2f}s')

Loading images: 100%|████████████████████████████████████████████████████████████| 4000/4000 [00:03<00:00, 1170.77it/s]


Loaded 20000 total images in 18.94s
Loaded 4000 total images in 18.96s


In [21]:
# Create arrays and specify labels
valid_indices_train = [i for i, img in enumerate(train_array)]
valid_indices_test = [i for i, img in enumerate(test_array)]
all_images_train = np.array(train_array)
all_images_test = np.array(test_array)
all_labels_train = onehot_labels_train[train_df['label_encoded']]
all_labels_test = onehot_labels_test[test_df['label_encoded']]

print('Final train dataset shape:', all_images_train.shape, all_labels_train.shape)
print('Final test dataset shape:', all_images_test.shape, all_labels_test.shape)

Final train dataset shape: (20000, 64, 64, 3) (20000, 4)
Final test dataset shape: (4000, 64, 64, 3) (4000, 4)


# Build cGAN

## Generator

In [22]:
def make_generator():
    noise_input = layers.Input(shape=(100,))
    label_input = layers.Input(shape=(4,))
    x = layers.Concatenate()([noise_input, label_input])
    x = layers.Dense(8*8*256)(x)
    x = layers.Reshape((8,8,256))(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(128, (4,4), strides=(2,2), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(64, (4,4), strides=(2,2), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', activation='tanh')(x)
    return tf.keras.Model([noise_input, label_input], x, name='Generator')

generator = make_generator()
generator.summary()

Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 100)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_1 (InputLayer)    │ (None, 4)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 104)               │               0 │ input_layer[0][0],         │
│                               │                           │                 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 16384)             │       1,720,320 │ concatenate[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape (Reshape)             │ (None, 8, 8, 256)         │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 8, 8, 256)         │           1,024 │ reshape[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu (LeakyReLU)       │ (None, 8, 8, 256)         │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_transpose              │ (None, 16, 16, 128)       │         524,416 │ leaky_re_lu[0][0]          │
│ (Conv2DTranspose)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 16, 16, 128)       │             512 │ conv2d_transpose[0][0]     │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_1 (LeakyReLU)     │ (None, 16, 16, 128)       │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_transpose_1            │ (None, 32, 32, 64)        │         131,136 │ leaky_re_lu_1[0][0]        │
│ (Conv2DTranspose)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 32, 32, 64)        │             256 │ conv2d_transpose_1[0][0]   │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_2 (LeakyReLU)     │ (None, 32, 32, 64)        │               0 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_transpose_2            │ (None, 64, 64, 3)         │           3,075 │ leaky_re_lu_2[0][0]        │
│ (Conv2DTranspose)             │                           │               

 Total params: 2,380,739 (9.08 MB)

 Trainable params: 2,379,843 (9.08 MB)

 Non-trainable params: 896 (3.50 KB)

## Discriminator

In [23]:
def make_discriminator():
    img_input = layers.Input(shape=(64,64,3))
    label_input = layers.Input(shape=(4,))
    label_img = layers.Dense(64*64*1)(label_input)
    label_img = layers.Reshape((64,64,1))(label_img)

    x = layers.Concatenate()([img_input, label_img])
    x = layers.Conv2D(64, (4,4), strides=(2,2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(256, (4,4), strides=(2,2), padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(1)(x)
    return tf.keras.Model([img_input, label_input], x, name='Discriminator')

discriminator = make_discriminator()
discriminator.summary()

Model: "Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)    │ (None, 4)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 4096)              │          20,480 │ input_layer_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 64, 64, 3)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reshape_1 (Reshape)           │ (None, 64, 64, 1)         │               0 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_1 (Concatenate)   │ (None, 64, 64, 4)         │               0 │ input_layer_2[0][0],       │
│                               │                           │                 │ reshape_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 32, 32, 64)        │           4,160 │ concatenate_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_3 (LeakyReLU)     │ (None, 32, 32, 64)        │               0 │ conv2d[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 32, 32, 64)        │               0 │ leaky_re_lu_3[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 16, 16, 256)       │         262,400 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ leaky_re_lu_4 (LeakyReLU)     │ (None, 16, 16, 256)       │               0 │ conv2d_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 16, 16, 256)       │               0 │ leaky_re_lu_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 65536)             │               0 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 1)                 │          65,537 │ flatten[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 352,577 (1.34 MB)

 Trainable params: 352,577 (1.34 MB)

 Non-trainable params: 0 (0.00 B)

## Training

In [24]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
gen_opt = tf.keras.optimizers.Adam(1e-4)
disc_opt = tf.keras.optimizers.Adam(1e-4)

@tf.function
def train_step(images, labels):
    noise = tf.random.normal([images.shape[0], 100])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator([noise, labels], training=True)
        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        gen_loss = cross_entropy(tf.ones_like(fake_output), fake_output)
        disc_loss = (cross_entropy(tf.ones_like(real_output), real_output)
                     + cross_entropy(tf.zeros_like(fake_output), fake_output))

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_opt.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    disc_opt.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    return gen_loss, disc_loss

## Training loop

In [25]:
# Image generator function
def generate_sample(label_text, ax):
    idx = le.transform([label_text])
    onehot = encoder.transform(idx.reshape(-1,1))
    onehot = tf.convert_to_tensor(onehot, dtype=tf.float32)
    noise = tf.random.normal([1,100])
    gen_img = generator([noise, onehot], training=False)
    ax.imshow((gen_img[0]+1)/2)
    ax.set_title(label_text)
    ax.axis('off')

In [ ]:
# Training loop
AUTOTUNE = tf.data.AUTOTUNE

dataset = tf.data.Dataset.from_tensor_slices((all_images_train, all_labels_train)).shuffle(1000).batch(64).prefetch(AUTOTUNE)

for epoch in range(50):
    start_epoch = time.time()
    for batch_images, batch_labels in tqdm(dataset, desc=f"Epoch {epoch+1}"):
        g_loss, d_loss = train_step(batch_images, batch_labels)

    print(f'Epoch {epoch+1}: Gen loss={g_loss.numpy():.4f}, Disc loss={d_loss.numpy():.4f} '
          f'in {time.time()-start_epoch:.2f}s')

    if (epoch+1) % 5 == 0:
        print(f'Showing generated samples at epoch {epoch+1}')
        fig, axes = plt.subplots(1, 4, figsize=(12,3))
        generate_sample('food', axes[0])
        generate_sample('inside', axes[1])
        generate_sample('outside', axes[2])
        generate_sample('drink', axes[3])
        plt.show()

Epoch 1:  20%|██████████████▍                                                         | 63/313 [01:59<10:53,  2.61s/it]

In [ ]:
# Show a Sample of generated images
fig, axes = plt.subplots(1, 4, figsize=(12,3))
generate_sample('food', axes[0])
generate_sample('inside', axes[1])
generate_sample('outside', axes[2])
generate_sample('drink', axes[3])
plt.show()

## Evaluation of cGAN

In [ ]:
def compute_fid_multisample(real_imgs, fake_imgs, batch_size=32):
    # Ensure images are float32 and in [-1,1] range (typical GAN)
    real_imgs = tf.convert_to_tensor(real_imgs, dtype=tf.float32)
    fake_imgs = tf.convert_to_tensor(fake_imgs, dtype=tf.float32)

    # Resize to (299,299,3) for Inception
    real_imgs_resized = tf.image.resize(real_imgs, (299,299)).numpy()
    fake_imgs_resized = tf.image.resize(fake_imgs, (299,299)).numpy()

    # Preprocess
    real_imgs_resized = preprocess_input(real_imgs_resized)
    fake_imgs_resized = preprocess_input(fake_imgs_resized)

    # Load Inception
    model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

    # Get activations
    act_real = model.predict(real_imgs_resized, batch_size=batch_size, verbose=0)
    act_fake = model.predict(fake_imgs_resized, batch_size=batch_size, verbose=0)

    # Compute means & covariances
    mu1, sigma1 = act_real.mean(axis=0), np.cov(act_real, rowvar=False)
    mu2, sigma2 = act_fake.mean(axis=0), np.cov(act_fake, rowvar=False)

    # FID calculation
    ssdiff = np.sum((mu1 - mu2)**2)
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = ssdiff + np.trace(sigma1 + sigma2 - 2*covmean)
    print(f"FID Score: {fid:.4f}")
    return fid

In [ ]:
def generate_test_images_and_evaluate(generator, real_imgs, real_labels, label_names, num_images=5):
    # InceptionV3 feature extractor
    inception = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

    plt.figure(figsize=(4 * num_images, 4))
    idxs = np.random.choice(len(real_imgs), size=num_images, replace=False)

    for i, idx in enumerate(idxs):
        real_img = real_imgs[idx]
        real_label = real_labels[idx]

        noise = tf.random.normal([1,100])
        fake_img = generator([noise, tf.convert_to_tensor(real_label.reshape(1,-1), dtype=tf.float32)], training=False)[0]

        # Resize to 299 for inception
        real_resized = tf.image.resize(real_img, (299,299)).numpy()
        fake_resized = tf.image.resize(fake_img, (299,299)).numpy()

        # Compute FID (single pair hack)
        fid = compute_fid_for_images(real_resized, fake_resized, inception)

        # Compute IS proxy (variance over activations)
        act_fake = inception.predict(preprocess_input(np.expand_dims(fake_resized,0)), verbose=0)
        is_score = np.exp(np.mean(np.log(1e-6 + np.var(act_fake, axis=0))))

        # Convert one-hot to label
        class_idx = real_label.argmax()
        label_text = label_names[class_idx]

        plt.subplot(1, num_images, i+1)
        plt.imshow((fake_img * 0.5 + 0.5))
        plt.title(f"Label: {label_text}\nFID: {fid:.2f} IS: {is_score:.2f}\nFAKE")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Generate 5 images with best FID and IS
generate_test_images_and_evaluate(generator, all_images_test, all_labels_test, LABELS, num_images=5)

## Generate 5 Best Images

In [ ]:
def generate_best_images_multi(generator, discriminator, labels_list, encoder, num_samples=500, top_k=1):
    plt.figure(figsize=(5*len(labels_list), 5))  # one column per label

    for i, label_name in enumerate(labels_list):
        # Encode label to onehot
        idx = encoder.transform([[label_name]])
        onehot = tf.convert_to_tensor(idx.repeat(num_samples, axis=0), dtype=tf.float32)

        # Sample noise and generate images
        noise = tf.random.normal([num_samples, 100])
        fake_imgs = generator([noise, onehot], training=False)

        # Get discriminator scores
        disc_scores = discriminator([fake_imgs, onehot], training=False).numpy().flatten()

        # Get top-k
        best_indices = np.argsort(-disc_scores)[:top_k]

        # For simplicity, just take the best one
        best_img = (fake_imgs[best_indices[0]].numpy() + 1) / 2

        plt.subplot(1, len(labels_list), i+1)
        plt.imshow(best_img)
        plt.title(f"'{label_name}'\nDisc: {disc_scores[best_indices[0]]:.2f}")
        plt.axis('off')

    plt.suptitle(f"Top {top_k} image per category by discriminator confidence")
    plt.show()

In [ ]:
# Generate Best Images
generate_best_images_multi(
    generator,
    discriminator,
    labels_list=[i for i in LABELS],
    encoder=encoder,
    num_samples=500,
    top_k=1
)